# TNT V3: language-pretrained encoder → M1 → Gold

This is the controlled downstream test of the unpaired Wolof encoder. It changes only encoder initialization. Architecture, the 3,330-row M1 sample, optimization, 20 synthetic epochs, 20 Gold epochs, and Gold-dev checkpoint selection match the completed V3 M1 control. Gold-test remains disabled.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling.tnt_hybrid_v3_language_pretraining_gate import (
    completed_language_encoder,
    control_runtime_reference,
    development_comparison,
    gate_protocol,
    load_transfer_data,
    prepare_transferred_initialization,
    run_language_gold_stage,
    run_language_synthetic_stage,
)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


CUDA: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 1. Reload the unchanged V3 protocol and data

The data-preparation function deterministically recreates the same matched M1 sample and locked Gold train/dev split as the random-encoder control.

In [2]:
protocol = gate_protocol()
gold, selected, dataset_audit = load_transfer_data(protocol)

print(protocol)
print('Gold:', {name: len(frame) for name, frame in gold.items()})
print('Selected M1 rows:', len(selected['m1']))
display(dataset_audit)

CompactBenchmarkProtocol(seed=2026, matched_unique_rows=3330, synthetic_epochs=20.0, gold_epochs=20.0, synthetic_learning_rate=0.0003, gold_learning_rate=0.0001, synthetic_batch_size=64, gold_batch_size=32, evaluation_batch_size=64, generated_eval_interval_epochs=5)
Gold: {'train': 142, 'dev': 30, 'test': 29}
Selected M1 rows: 3330


,method,loaded_rows,unique_pair_rows,unique_target_rows,selected_rows,common_target_pool,seed,dev_input_overlap,dev_pair_overlap,test_input_overlap,test_pair_overlap,source_over_255_chars,target_over_255_chars
0,m1,17777,16377,16375,3330,3430,2026,0,0,0,0,8,6
1,m2,17777,16544,16375,3330,3430,2026,0,0,0,0,10,6
2,m3,17777,16384,16375,3330,3430,2026,0,0,0,0,8,6
3,m4,3433,3433,3430,3330,3430,2026,0,0,0,0,7,6
4,m6,17777,16427,16375,3330,3430,2026,0,0,0,0,6,6


## 2. Verify the completed language encoder

Transfer is permitted only when encoder pretraining is complete and its separate unpaired test has been evaluated.

In [3]:
encoder_checkpoint, encoder_vocabulary, encoder_manifest = completed_language_encoder()
print('Encoder:', encoder_checkpoint)
print('Pretraining test evaluated:', encoder_manifest['test_evaluated'])
print('Pretraining test metrics:', json.dumps(encoder_manifest['test_metrics'], indent=2))

Encoder: C:\Users\thioy\Desktop\School\PFE\checkpoints\wolof_encoder_pretraining_v1\476eeb46caee\best_encoder.pt
Pretraining test evaluated: True
Pretraining test metrics: {
  "loss": 2.6500535460557635,
  "masked_accuracy": 0.2698189712972127,
  "masked_characters": 31321,
  "rows": 3018
}


## 3. Create or reuse the transferred initialization

All 120 downstream vocabulary rows must be found in the 272-character pretraining vocabulary. Normalization-operation, replacement, boundary, word, and segment-length heads retain the same seeded random weights as the earlier control.

In [4]:
initialization = prepare_transferred_initialization(gold, selected, protocol)
transfer_manifest = json.loads(
    (initialization / 'initialization_manifest.json').read_text(encoding='utf-8')
)
print('Initialization:', initialization)
print(json.dumps(transfer_manifest['transfer_audit'], indent=2))

[TNT-V3] Reusing initialization: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\shared_random_base\tnt_hybrid_v3_char_dadd682dc9cc
[language transfer] Reusing C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\language_pretrained_initialization\1e2a3912778b
Initialization: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\language_pretrained_initialization\1e2a3912778b
{
  "encoder_tensors_copied": 36,
  "position_tensors_copied": 1,
  "target_vocabulary_size": 120,
  "source_vocabulary_size": 272,
  "embedding_rows_copied": 120,
  "embedding_rows_missing": 0,
  "missing_target_tokens": [],
  "exact_encoder_and_position_parameters": 1383744,
  "mapped_embedding_parameters": 23040,
  "total_transferred_parameters": 1406784
}


## 4. Runtime reference

Unlike the small masked-character pretrainer, this uses the existing Hugging Face TNT normalization pipeline. On the previous V3 run, synthetic M1 took about 110 minutes and Gold fine-tuning about 12 minutes. The cells are separated and resumable.

In [5]:
print(json.dumps(control_runtime_reference(), indent=2))

{
  "synthetic_pretrain": 110.42079833333334,
  "gold_finetune_continuous": 11.963973333333334,
  "total": 122.38477166666667
}


## 5. M1 synthetic pretraining

This is the long stage. Training and generated Gold-dev evaluations remain visible. If interrupted, rerun the cell with `rerun=False`; the existing attempt resumes or is reused when complete.

In [6]:
RUN_M1_SYNTHETIC = True
if RUN_M1_SYNTHETIC:
    synthetic_record = run_language_synthetic_stage(
        gold, selected, initialization, protocol, rerun=False
    )
    print('Synthetic checkpoint:', synthetic_record['best_model_path'])
    print('Synthetic dev metrics:', json.dumps(synthetic_record['development_metrics'], indent=2))


[M7] START m1 / synthetic_pretrain / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\language_pretrained_initialization\1e2a3912778b


You are using a model of type `wolof_tnt_hybrid_v3_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

[M7] Loaded 1,783,707 total parameters; training 1,783,707 (100.0000%) via full_finetune
[M7] Tokenizing 3,330 training and 30 development pairs


Tokenizing TNT edit pairs:   0%|          | 0/3330 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridV3TransformerForConditionalGeneration.forward` and have been ignored: source_id, target_text, source_text. If source_id, target_text, source_text are not expected by `TntHybridV3TransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m1 / synthetic_pretrain / seed 2026 / attempt 001; dev CER=0.19150521609538004
Synthetic checkpoint: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model
Synthetic dev metrics: {
  "rows": 30,
  "cer": 0.19150521609538004,
  "wer": 0.5966101694915255,
  "chrf": 59.64092075657178,
  "exact_match": 0.0,
  "correction_tp": 3,
  "correction_fp": 72,
  "correction_fn": 69,
  "correction_precision": 0.04,
  "correction_recall": 0.041666666666666664,
  "correction_f1": 0.04081632653061224,
  "overcorrection_rate": 0.96,
  "unchanged_reference_rows": 0,
  "unchanged_sentence_overcorrection_rate": 0.0
}


## 6. Gold-train fine-tuning

Run only after the synthetic stage reports complete. Restarting the notebook is safe: the cell reloads the completed synthetic stage before beginning or resuming Gold fine-tuning.

In [7]:
RUN_GOLD_FINETUNE = True
if RUN_GOLD_FINETUNE:
    if 'synthetic_record' not in globals():
        synthetic_record = run_language_synthetic_stage(
            gold, selected, initialization, protocol, rerun=False
        )
    gold_selection = run_language_gold_stage(
        gold, synthetic_record, protocol, rerun=False
    )
    print(json.dumps(gold_selection['development_metrics'], indent=2))


[M7] START m1 / gold_finetune_continuous / seed 2026 / attempt 001 (trained)
[M7] Output: C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\gold_finetune_continuous\attempt_001
[M7] Loading C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model


loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model\config.json
loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model\config.json
You are using a model of type `wolof_tnt_hybrid_v3_transformer` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Model config PreTrainedConfig {
  "architectures": [
    "TntHybridV3TransformerForConditionalGeneration"
  ],
  "attention_heads": 4,
  "bos_token_id": 1,
  "boundary_class_weights": [
    0.5533591001386836,
    1.

Loading weights:   0%|          | 0/51 [00:00<?, ?it/s]

loading configuration file C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\synthetic_pretrain\attempt_001\best_model\generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 1,
  "decoder_start_token_id": 1,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": [
    2
  ],
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 1.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 0,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 1,
  "num_return_sequences": 1,
  "output_attentions": false,
  "output_hidden_states": false,
  "output_scores": false,
  "pad_token_id": 0,
  "remove_invalid_values": false,
  "repetition_penalty"

[M7] Loaded 1,783,707 total parameters; training 1,783,707 (100.0000%) via full_finetune
[M7] Tokenizing 142 training and 30 development pairs


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\model.safetensors
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json
tokenizer config file saved in C:\Users\thioy\Desktop\School\PFE\checkpoints\m7_seq2seq\tnt_hybrid_v3_language_pretrained_m1_gate_v1\m1\seed_2026\gold_finetune_continuous\attempt_001\best_model\tokenizer_config.json


[M7] Running development prediction and final metrics


The following columns in the test set don't have a corresponding argument in `TntHybridV3TransformerForConditionalGeneration.forward` and have been ignored: source_id, target_text, source_text. If source_id, target_text, source_text are not expected by `TntHybridV3TransformerForConditionalGeneration.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 30
  Batch size = 64


[M7] COMPLETE m1 / gold_finetune_continuous / seed 2026 / attempt 001; dev CER=0.1870342771982116
[compact benchmark] m1: continuous 20-epoch gold run complete; selected dev CER=0.1870342771982116
{
  "rows": 30,
  "cer": 0.1870342771982116,
  "wer": 0.6440677966101694,
  "chrf": 63.07658337289543,
  "exact_match": 0.0,
  "correction_tp": 3,
  "correction_fp": 74,
  "correction_fn": 69,
  "correction_precision": 0.03896103896103896,
  "correction_recall": 0.041666666666666664,
  "correction_f1": 0.040268456375838924,
  "overcorrection_rate": 0.961038961038961,
  "unchanged_reference_rows": 0,
  "unchanged_sentence_overcorrection_rate": 0.0
}


## 7. Controlled Gold-dev comparison

The decisive comparison is against the completed random-encoder V3 M1 run. Lower CER/WER and overcorrection, and higher chrF/correction F1, indicate improvement. This cell never opens Gold-test.

In [8]:
comparison = development_comparison(gold['dev'])
display(comparison)

,condition,initialization,rows,cer,wer,chrf,exact_match,correction_tp,correction_fp,correction_fn,correction_precision,correction_recall,correction_f1,overcorrection_rate,unchanged_reference_rows,unchanged_sentence_overcorrection_rate
0,identity,none,30,0.233234,0.630508,56.052143,0.0,0,0,72,0.000000,0.000000,0.000000,0.000000,0,0.0
1,m1_v3_random_encoder,seeded random,30,0.194486,0.691525,59.948595,0.0,2,69,70,0.028169,0.027778,0.027972,0.971831,0,0.0
2,m1_v3_language_pretrained_encoder,unpaired formal+informal MLM,30,0.187034,0.644068,63.076583,0.0,3,74,69,0.038961,0.041667,0.040268,0.961039,0,0.0
